# setup + imports

In [1]:
import json
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

/Users/ecaterina/practice/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1103 03:22:01.634000 59102 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [2]:
train = pd.read_csv('train.csv')
val = pd.read_csv('val.csv')
test = pd.read_csv('test.csv')
hints = pd.read_csv('hint_descriptions.csv')

In [5]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 6123.68it/s]


## embeddings

In [20]:
hintvec = {}
for i in range(len(hints)):
    hintid = int(hints['hintID'].iloc[i])
    hintvec[hintid] = model.encode(str(hints['description'].iloc[i]), normalize_embeddings=True)

allopts = []
for data in [train, val, test]:
    for opts in data['options_json']:
        for w in json.loads(opts):
            if w not in allopts:
                allopts.append(w)

vecs = model.encode(allopts, normalize_embeddings=True)
optvecs = {}
for i in range(len(allopts)):
    optvecs[allopts[i]] = vecs[i]


## ranking

In [30]:
def rankeaza(hids, opts):
    scores = []
    for w in opts:
        sc = 0
        for h in hids:
            sc += hintvec[h] @ optvecs[w]
        scores.append(sc)
    order = np.argsort(scores)[::-1]
    top10 = []
    for k in order:
        w = opts[k]
        if w not in top10:
            top10.append(w)
        if len(top10)==10:
            break
    return top10

## validare

In [31]:
valsc = 0
for i in range(len(val)):
    top10 = rankeaza(json.loads(val['hints_json'].iloc[i]), json.loads(val['options_json'].iloc[i]))
    if val['answer'].iloc[i] in top10:
        valsc += 0.9
        if val['answer'].iloc[i] == top10[0]:
            valsc += 0.1
print('answer in top 10:', valsc / len(val))

answer in top 10: 0.5974683544303794


## submission

In [32]:
rows = []
for i in range(len(test)):
    hint_ids = json.loads(test['hints_json'].iloc[i])
    options = json.loads(test['options_json'].iloc[i])
    ranked = rankeaza(hint_ids, options)
    rows.append((test['datapointID'].iloc[i], 1, '|'.join(ranked)))
pd.DataFrame(rows, columns=['datapointID', 'subtaskID', 'answer']).to_csv('submission.csv', index=False)